# Policy Ranking Agreement (Q3)

Reads compact provenance from `outputs/in_distribution/policy_ranking_agreement/`.

**Question**: Does a user simulator rank recommender policies in the same order as real held-out data?

**Protocol**: Each policy generates a top-k recommendation list per evaluation user. A scorer assigns a simulated score to every recommended item. Mean simulated score becomes the policy's simulated utility; mean real target (held-out hit rate) becomes the real utility. Kendall's τ and Spearman's ρ measure agreement between the simulated and real policy rankings.

**Main metric**: `test.kendall_tau`. Spearman's ρ is reported alongside as a secondary measure. Both are `None` when fewer than 3 policies are evaluated (undefined for K < 3 — see `warning` field).

**Sections**:
1. Setup and artifact collection
2. Per-run results table
3. Policy utilities (simulated vs. real, per run)
4. Seed-averaged rank correlations
5. Verifications

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists() and candidate.name == "beyond-click-sim":
            return candidate
    raise RuntimeError("Could not find beyond-click-sim repo root")


REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "outputs" / "in_distribution" / "policy_ranking_agreement"

METHODS = {
    "popularity_scorer": "Popularity scorer",
}
DATASET_LABELS = {"ml-1m": "ML-1M", "steam": "Steam"}

TASK_RE = re.compile(
    r"^(?P<dataset>ml-1m|steam)_policy_ranking"
    r"(?:_eval_users(?P<eval_users>\d+))?"
    r"_seed(?P<seed>\d+)$"
)
RUN_TS_RE = re.compile(r"^(?P<timestamp>\d{8}T\d{6}Z)_")

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"RESULTS_ROOT: {RESULTS_ROOT}")
print(f"Results directory exists: {RESULTS_ROOT.exists()}")
if RESULTS_ROOT.exists():
    run_dirs = [p for p in RESULTS_ROOT.iterdir() if p.is_dir()]
    print(f"Run directories found: {len(run_dirs)}")

In [ ]:
def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def as_dict(value: Any) -> dict[str, Any]:
    return value if isinstance(value, dict) else {}


def as_list(value: Any) -> list:
    return value if isinstance(value, list) else []


def timestamp_from_run_dir(run_dir: Path) -> str:
    match = RUN_TS_RE.match(run_dir.name)
    return "" if match is None else match.group("timestamp")


def parse_task_name(task: str) -> dict[str, Any]:
    match = TASK_RE.match(task)
    if match is None:
        return {"dataset": "unknown", "eval_users": None, "seed": -1}
    return {
        "dataset": match.group("dataset"),
        "eval_users": int(match.group("eval_users")) if match.group("eval_users") else None,
        "seed": int(match.group("seed")),
    }


def float_or_none(value: Any) -> float | None:
    return None if value is None else float(value)

In [ ]:
def collect_runs() -> pd.DataFrame:
    """Scan RESULTS_ROOT for policy ranking agreement artifacts.

    Returns one row per (task, method, run directory). If multiple runs exist
    for the same (task, method), all are kept — use `latest_runs` to keep only
    the most recent.
    """
    if not RESULTS_ROOT.exists():
        print(f"WARNING: {RESULTS_ROOT} does not exist. No runs found.")
        print("Run the runner first:")
        print("  uv run python -m runners.in_distribution.policy_ranking_agreement.run")
        return pd.DataFrame()

    rows: list[dict[str, Any]] = []
    for metrics_path in sorted(RESULTS_ROOT.glob("*/metrics.json")):
        run_dir = metrics_path.parent
        manifest_path = run_dir / "manifest.json"
        predictions_path = run_dir / "predictions.parquet"

        metrics = read_json(metrics_path)
        if metrics.get("protocol") != "policy_ranking":
            continue

        method = str(metrics.get("method", ""))
        task = str(metrics.get("task", ""))
        task_info = parse_task_name(task)
        test_block = as_dict(metrics.get("test"))
        manifest = read_json(manifest_path) if manifest_path.exists() else {}
        task_manifest = as_dict(as_dict(manifest.get("task")).get("manifest"))
        policies_manifest = as_list(task_manifest.get("policies"))
        policy_names_manifest = [p.get("name") for p in policies_manifest]

        rows.append({
            "timestamp": timestamp_from_run_dir(run_dir),
            "run": run_dir.name,
            "task": task,
            "method": method,
            "method_label": METHODS.get(method, method),
            "dataset": task_info["dataset"],
            "eval_users": task_info["eval_users"],
            "seed": task_info["seed"],
            "n_policies": test_block.get("n_policies"),
            "kendall_tau": float_or_none(test_block.get("kendall_tau")),
            "kendall_tau_pvalue": float_or_none(test_block.get("kendall_tau_pvalue")),
            "spearman_rho": float_or_none(test_block.get("spearman_rho")),
            "spearman_rho_pvalue": float_or_none(test_block.get("spearman_rho_pvalue")),
            "simulated_rank": as_list(test_block.get("simulated_rank")),
            "real_rank": as_list(test_block.get("real_rank")),
            "simulated_utilities": as_dict(test_block.get("simulated_utilities")),
            "real_utilities": as_dict(test_block.get("real_utilities")),
            "warning": test_block.get("warning"),
            "utility_aggregation": metrics.get("utility_aggregation"),
            "policy_names_manifest": policy_names_manifest,
            "has_predictions": predictions_path.exists(),
            "train_rows": as_dict(task_manifest.get("rows")).get("train"),
            "test_rows": as_dict(task_manifest.get("rows")).get("test"),
            "train_users": as_dict(task_manifest.get("users")).get("train"),
            "test_users": as_dict(task_manifest.get("users")).get("test"),
        })

    if not rows:
        print("No policy ranking agreement metrics.json files found in RESULTS_ROOT.")
        print("Run the runner first:")
        print("  uv run python -m runners.in_distribution.policy_ranking_agreement.run")
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values(["dataset", "method", "seed", "timestamp"])
        .reset_index(drop=True)
    )


all_runs = collect_runs()
if all_runs.empty:
    print("No runs found — remaining cells will produce empty output.")
else:
    print(f"Total runs collected: {len(all_runs)}")
    display(all_runs[["dataset", "method", "seed", "timestamp", "run"]].head(20))

In [ ]:
# Keep the most recent run per (task, method) — dedup by latest timestamp.
if not all_runs.empty:
    latest_runs = (
        all_runs
        .sort_values(["task", "method", "timestamp", "run"])
        .groupby(["task", "method"], as_index=False, dropna=False)
        .tail(1)
        .sort_values(["dataset", "method", "seed"])
        .reset_index(drop=True)
    )
else:
    latest_runs = all_runs.copy()

## Per-Run Results Table

One row per (task, method). `kendall_tau` and `spearman_rho` are `None` when fewer than 3 policies are evaluated — check the `warning` column. `simulated_rank` and `real_rank` show the full policy ordering (from best to worst) under each perspective.

In [ ]:
if not latest_runs.empty:
    display_cols = [
        "dataset", "method_label", "seed",
        "n_policies", "kendall_tau", "kendall_tau_pvalue",
        "spearman_rho", "spearman_rho_pvalue",
        "simulated_rank", "real_rank",
        "warning", "train_rows", "test_rows", "test_users", "run",
    ]
    display_cols = [c for c in display_cols if c in latest_runs.columns]

    display(
        latest_runs[display_cols].style
        .format({
            "kendall_tau": "{:.4f}",
            "kendall_tau_pvalue": "{:.4f}",
            "spearman_rho": "{:.4f}",
            "spearman_rho_pvalue": "{:.4f}",
        }, na_rep="--")
        .hide(axis="index")
    )

## Policy Utilities (Simulated vs. Real)

For each run, show the per-policy mean simulated score (what the scorer thinks of those recommendations) and the per-policy mean real target (actual held-out hit rate). The agreement between the orderings of these two columns is what Kendall's τ and Spearman's ρ capture.

In [ ]:
def expand_utilities(runs: pd.DataFrame) -> pd.DataFrame:
    """Expand simulated_utilities and real_utilities dicts into long-format rows."""
    rows = []
    for _, run in runs.iterrows():
        sim_util = run["simulated_utilities"]
        real_util = run["real_utilities"]
        all_policies = sorted(set(list(sim_util.keys()) + list(real_util.keys())))
        for policy in all_policies:
            rows.append({
                "dataset": run["dataset"],
                "method_label": run["method_label"],
                "seed": run["seed"],
                "policy": policy,
                "simulated_utility": sim_util.get(policy),
                "real_utility": real_util.get(policy),
                "simulated_rank_position": (
                    run["simulated_rank"].index(policy) + 1
                    if policy in run["simulated_rank"] else None
                ),
                "real_rank_position": (
                    run["real_rank"].index(policy) + 1
                    if policy in run["real_rank"] else None
                ),
            })
    return pd.DataFrame(rows)


if not latest_runs.empty:
    utilities_df = expand_utilities(latest_runs)
    display(
        utilities_df.style
        .format({
            "simulated_utility": "{:.6f}",
            "real_utility": "{:.6f}",
        }, na_rep="--")
        .hide(axis="index")
    )

## Seed-Averaged Rank Correlations

Mean and std of Kendall's τ and Spearman's ρ across seeds, grouped by (dataset, method). Only includes seeds where both metrics are non-null (i.e. K ≥ 3 policies). Seeds where K < 3 are counted but excluded from the average.

In [ ]:
if not latest_runs.empty:
    valid = latest_runs.dropna(subset=["kendall_tau", "spearman_rho"])
    excluded = len(latest_runs) - len(valid)
    if excluded > 0:
        print(f"NOTE: {excluded} run(s) excluded from seed average (K < 3 policies, tau/rho undefined).")

    if valid.empty:
        print("No runs with K ≥ 3 policies to average.")
    else:
        summary = (
            valid
            .groupby(["dataset", "method_label"], dropna=False)
            .agg(
                n_seeds=("seed", "nunique"),
                seeds=("seed", lambda s: tuple(sorted(int(v) for v in s))),
                kendall_tau_mean=("kendall_tau", "mean"),
                kendall_tau_std=("kendall_tau", "std"),
                spearman_rho_mean=("spearman_rho", "mean"),
                spearman_rho_std=("spearman_rho", "std"),
                kendall_tau_pvalue_mean=("kendall_tau_pvalue", "mean"),
                spearman_rho_pvalue_mean=("spearman_rho_pvalue", "mean"),
            )
            .reset_index()
        )
        display(
            summary.style
            .format({
                "kendall_tau_mean": "{:.4f}",
                "kendall_tau_std": "{:.4f}",
                "spearman_rho_mean": "{:.4f}",
                "spearman_rho_std": "{:.4f}",
                "kendall_tau_pvalue_mean": "{:.4f}",
                "spearman_rho_pvalue_mean": "{:.4f}",
            }, na_rep="--")
            .hide(axis="index")
        )

## Verifications

Structural integrity checks on the collected artifacts. Each check should produce no failures on well-formed runs.

In [ ]:
def check_policy_name_consistency(runs: pd.DataFrame) -> None:
    """Verify that policy names in metrics match those declared in the manifest."""
    failures = []
    for _, run in runs.iterrows():
        manifest_names = set(run["policy_names_manifest"])
        metrics_names = set(run["simulated_utilities"].keys())
        if manifest_names and metrics_names and manifest_names != metrics_names:
            failures.append({
                "run": run["run"],
                "manifest_policies": sorted(manifest_names),
                "metrics_policies": sorted(metrics_names),
            })
    if failures:
        print("FAIL: Policy name mismatch between manifest and metrics:")
        for f in failures:
            print(f"  run={f['run']}")
            print(f"    manifest: {f['manifest_policies']}")
            print(f"    metrics:  {f['metrics_policies']}")
    else:
        print(f"OK: All {len(runs)} runs have consistent policy names in manifest and metrics.")


if not latest_runs.empty:
    check_policy_name_consistency(latest_runs)

In [ ]:
def check_rank_order_consistent_with_utilities(runs: pd.DataFrame) -> None:
    """Verify that simulated_rank is the descending sort of simulated_utilities,
    and real_rank is the descending sort of real_utilities.
    """
    failures = []
    for _, run in runs.iterrows():
        sim_util = run["simulated_utilities"]
        real_util = run["real_utilities"]
        sim_rank = run["simulated_rank"]
        real_rank = run["real_rank"]
        if not sim_util or not real_util:
            continue

        expected_sim_rank = sorted(sim_util, key=sim_util.__getitem__, reverse=True)
        expected_real_rank = sorted(real_util, key=real_util.__getitem__, reverse=True)

        if sim_rank != expected_sim_rank:
            failures.append({"run": run["run"], "kind": "simulated", "got": sim_rank, "expected": expected_sim_rank})
        if real_rank != expected_real_rank:
            failures.append({"run": run["run"], "kind": "real", "got": real_rank, "expected": expected_real_rank})

    if failures:
        print("FAIL: Rank order inconsistent with utilities:")
        for f in failures:
            print(f"  run={f['run']} kind={f['kind']}")
            print(f"    got:      {f['got']}")
            print(f"    expected: {f['expected']}")
    else:
        print(f"OK: All rank orders are consistent with utility values in all {len(runs)} runs.")


if not latest_runs.empty:
    check_rank_order_consistent_with_utilities(latest_runs)

In [ ]:
def check_predictions_parquet(runs: pd.DataFrame, results_root: Path) -> None:
    """Verify that predictions.parquet has expected columns and no 'prediction' column.

    Q3 is not a classification task — 'prediction' has no meaning here.
    Required columns: split, user_id, item_id, target, score, policy, rank.
    """
    required_cols = {"split", "user_id", "item_id", "target", "score", "policy", "rank"}
    failures = []

    for _, run in runs.iterrows():
        preds_path = results_root / run["run"] / "predictions.parquet"
        if not preds_path.exists():
            failures.append({"run": run["run"], "issue": "predictions.parquet missing"})
            continue
        preds = pd.read_parquet(preds_path)
        if "prediction" in preds.columns:
            failures.append({"run": run["run"], "issue": "has 'prediction' column (Q3 is not classification)"})
        missing = required_cols - set(preds.columns)
        if missing:
            failures.append({"run": run["run"], "issue": f"missing columns: {sorted(missing)}"})
        splits = set(preds["split"].unique())
        if splits != {"test"}:
            failures.append({"run": run["run"], "issue": f"unexpected split values: {splits} (expected only 'test')"})

    if failures:
        print("FAIL: predictions.parquet issues:")
        for f in failures:
            print(f"  run={f['run']}: {f['issue']}")
    else:
        n = sum(1 for _, r in runs.iterrows() if r["has_predictions"])
        print(f"OK: {n} predictions.parquet files have correct columns and split values.")


if not latest_runs.empty:
    check_predictions_parquet(latest_runs, RESULTS_ROOT)

In [ ]:
def check_no_duplicate_seeds(runs: pd.DataFrame) -> None:
    """Verify that latest_runs has at most one run per (dataset, method, seed) — no silent duplicates."""
    dupes = (
        runs.groupby(["dataset", "method", "seed"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )
    dupes = dupes[dupes["count"] > 1]
    if not dupes.empty:
        print("FAIL: Duplicate (dataset, method, seed) after deduplication:")
        display(dupes)
    else:
        print(f"OK: No duplicate (dataset, method, seed) entries in latest_runs ({len(runs)} runs).")


if not latest_runs.empty:
    check_no_duplicate_seeds(latest_runs)

In [ ]:
def check_utility_completeness(runs: pd.DataFrame) -> None:
    """Verify that simulated_utilities and real_utilities contain the same policy names."""
    failures = []
    for _, run in runs.iterrows():
        sim_keys = set(run["simulated_utilities"].keys())
        real_keys = set(run["real_utilities"].keys())
        if sim_keys != real_keys:
            failures.append({
                "run": run["run"],
                "sim_only": sorted(sim_keys - real_keys),
                "real_only": sorted(real_keys - sim_keys),
            })
    if failures:
        print("FAIL: Mismatched policy names between simulated and real utilities:")
        for f in failures:
            print(f"  run={f['run']}")
            if f["sim_only"]:
                print(f"    in simulated only: {f['sim_only']}")
            if f["real_only"]:
                print(f"    in real only: {f['real_only']}")
    else:
        print(f"OK: simulated_utilities and real_utilities cover the same policy names in all {len(runs)} runs.")


if not latest_runs.empty:
    check_utility_completeness(latest_runs)

In [ ]:
def check_protocol_field(runs: pd.DataFrame, results_root: Path) -> None:
    """Verify that manifest.json has protocol='policy_ranking' in all runs."""
    failures = []
    for _, run in runs.iterrows():
        manifest_path = results_root / run["run"] / "manifest.json"
        if not manifest_path.exists():
            failures.append({"run": run["run"], "issue": "manifest.json missing"})
            continue
        manifest = read_json(manifest_path)
        protocol = manifest.get("protocol")
        if protocol != "policy_ranking":
            failures.append({"run": run["run"], "issue": f"protocol={protocol!r} (expected 'policy_ranking')"})
    if failures:
        print("FAIL:")
        for f in failures:
            print(f"  run={f['run']}: {f['issue']}")
    else:
        print(f"OK: All {len(runs)} manifests have protocol='policy_ranking'.")


if not latest_runs.empty:
    check_protocol_field(latest_runs, RESULTS_ROOT)

In [ ]:
def check_k2_warning_present(runs: pd.DataFrame) -> None:
    """When K=2, tau/rho should be None and a warning should be set."""
    failures = []
    for _, run in runs.iterrows():
        if run["n_policies"] == 2:
            if run["kendall_tau"] is not None:
                failures.append({"run": run["run"], "issue": "kendall_tau is not None for K=2"})
            if not run["warning"]:
                failures.append({"run": run["run"], "issue": "warning is empty/None for K=2"})
    if failures:
        print("FAIL: K=2 invariant violated:")
        for f in failures:
            print(f"  run={f['run']}: {f['issue']}")
    else:
        k2_runs = (runs["n_policies"] == 2).sum() if not runs.empty else 0
        if k2_runs:
            print(f"PASS: {k2_runs} K=2 run(s) have tau=None and a warning set (as expected).")
        else:
            print("INFO: No K=2 runs found (all runs have K≥3, tau/rho are defined).")


if not latest_runs.empty:
    check_k2_warning_present(latest_runs)

## Coverage Summary

Which (dataset, method, seed) combinations have completed runs? Gaps indicate missing experiments.

In [ ]:
if not latest_runs.empty:
    expected_datasets = ["ml-1m", "steam"]
    expected_seeds = list(range(5))
    expected_methods = list(METHODS.keys())

    coverage_rows = []
    for dataset in expected_datasets:
        for method in expected_methods:
            for seed in expected_seeds:
                match = latest_runs[
                    (latest_runs["dataset"] == dataset)
                    & (latest_runs["method"] == method)
                    & (latest_runs["seed"] == seed)
                ]
                if match.empty:
                    coverage_rows.append({"dataset": dataset, "method": METHODS.get(method, method), "seed": seed, "status": "MISSING", "run": ""})
                else:
                    row = match.iloc[0]
                    tau = row["kendall_tau"]
                    tau_str = f"{tau:.4f}" if tau is not None else "K<3"
                    coverage_rows.append({"dataset": dataset, "method": METHODS.get(method, method), "seed": seed, "status": f"OK (τ={tau_str})", "run": row["run"]})

    coverage_df = pd.DataFrame(coverage_rows)

    missing = (coverage_df["status"] == "MISSING").sum()
    if missing > 0:
        print(f"WARNING: {missing} (dataset, method, seed) combination(s) missing.")

    display(
        coverage_df.style
        .apply(lambda col: [
            "color: red; font-weight: bold" if v == "MISSING" else ""
            for v in col
        ], subset=["status"])
        .hide(axis="index")
    )